# Tuning federated learnings
This notebook demonstrates how to tune a federated learning system using Flower.
We will configure the training process to use different hyperparameters across rounds,
allowing for more flexible and efficient federated training.


Key concepts covered:
- **Dynamic configuration**: Adjusting training parameters (like local epochs) per round
- **Client-side training**: How clients receive and apply configuration from the server
- **FedAvg strategy**: The federated averaging algorithm for aggregating model updates

**Requirements**: python3.10

In [1]:
# well flower framework seems to need the HF_TOKEN
import os
os.environ["HF_TOKEN"] = "hf_z**SHHSZ"
from huggingface_hub import whoami
# whoami()  # Confirms auth

In [2]:
import logging
import warnings
from logging import INFO, ERROR, LogRecord, DEBUG
from typing import List, Tuple, Dict, Optional, Union
from collections import OrderedDict

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets
from torchvision.transforms import Compose, ToTensor, Normalize
from torch.utils.data import DataLoader

from flwr.client import Client, ClientApp, NumPyClient
from flwr.server import ServerApp, ServerConfig, ServerAppComponents
from flwr.server.strategy import FedAvg
from flwr.simulation import run_simulation
from flwr_datasets import FederatedDataset
from flwr.common import ndarrays_to_parameters, Context, Metrics
from flwr.common.logger import (
    ConsoleHandler,
    console_handler,
    FLOWER_LOGGER,
    LOG_COLORS,
    log,
)

In [3]:
class InfoFilter(logging.Filter):
    def filter(self, record):
        return record.levelno == INFO

FLOWER_LOGGER.removeHandler(console_handler)
warnings.filterwarnings("ignore")

# To filter logging coming from the Simulation Engine
# so it's more readable in notebooks
backend_setup = {"init_args": {"logging_level": ERROR, "log_to_driver": True}}

class ConsoleHandlerV2(ConsoleHandler):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def format(self, record: LogRecord) -> str:
        """Format function that adds colors to log level."""
        if self.json:
            log_fmt = "{lv1='%(levelname)s', time='%(asctime)s', msg='%(message)s'}"
        else:
            log_fmt = (
                f"{LOG_COLORS[record.levelname] if self.colored else ''}"
                f"%(levelname)s {'%(asctime)s' if self.timestamps else ''}"
                f"{LOG_COLORS['RESET'] if self.colored else ''}"
                f": %(message)s"

            )
        formatter = logging.Formatter(log_fmt)
        return formatter.format(record)

console_handlerv2 = ConsoleHandlerV2(
    timestamps=False,
    json=False,
    colored=True,
)
console_handlerv2.setLevel(INFO)
console_handlerv2.addFilter(InfoFilter())
FLOWER_LOGGER.addHandler(console_handlerv2)

DEVICE = "cpu"

#### Prepare the datasets
We use `flwr-datasets` to load the MNIST dataset partitioned across 5 clients.
The data is split into train and test sets, with normalization applied.
* Prepare data using Flower Datasets.
Use `flwr-datasets` that provides with a Federated Dataset abstraction.

In [4]:
transforms = Compose([ToTensor(), Normalize((0.5,), (0.5,))])
def normalize(batch):
    batch["image"] = [transforms(img) for img in batch["image"]]
    return batch
                      

def load_data(partition_id):
    fds = FederatedDataset(dataset="mnist", partitioners={"train": 5})
    partition = fds.load_partition(partition_id)

    traintest = partition.train_test_split(test_size=0.2, seed=42)
    traintest = traintest.with_transform(normalize)
    trainset, testset = traintest["train"], traintest["test"]

    trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
    testloader = DataLoader(testset, batch_size=64)
    return trainloader, testloader

#### Clients configuration
Flower allows the server to send configuration values to clients at each round.
Here, we define a `fit_config` function that returns a dictionary with `local_epochs`,
which varies based on the server round (fewer epochs early, more later).

* Define fit_config.
Flower can send configuration values to clients.

In [5]:
def fit_config(server_round: int):
    config_dict = {
        "local_epochs": 2 if server_round < 3 else 5,
    }
    return config_dict

### Model Definition and Helper Functions

We define a simple fully connected neural network for MNIST classification.
Helper functions `get_weights` and `set_weights` convert between PyTorch state dicts
and NumPy arrays for federated learning communication.

In [6]:
class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.fc = nn.Linear(784, 128)
        self.relu = nn.ReLU()
        self.out = nn.Linear(128, 10)

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = self.fc(x)
        x = self.relu(x)
        x = self.out(x)
        return x

def get_weights(net):
    ndarrays = [
        val.cpu().numpy() for _, val in net.state_dict().items()
    ]
    return ndarrays

def set_weights(net, parameters):
    params_dict = zip(net.state_dict().keys(), parameters)
    state_dict = OrderedDict(
        {k: torch.tensor(v) for k, v in params_dict}
    )
    net.load_state_dict(state_dict, strict=True)

def train_model(model, trainloader, epochs):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

    model.train()
    for epoch in range(epochs):
        log(INFO, f"Running client {epochs} epochs")
        for batch in trainloader:
            images = batch["image"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

def evaluate_model(model, test_loader):
    model.eval()
    correct, loss = 0, 0.0
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for batch in test_loader:
            images = batch["image"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            outputs = model(images)
            labels = labels.to(DEVICE)
            loss += criterion(outputs, labels).item()
            correct += (
                (torch.max(outputs.data, 1)[1] == labels).sum().item()
            )
    accuracy = correct / len(testloader.dataset)
    return float(loss), float(accuracy)


### Server Strategy

The `FedAvg` strategy is configured with:
- `min_fit_clients=5`: Minimum number of clients required for training
- `fraction_evaluate=0.0`: Disable evaluation (optional)
- `on_fit_config_fn=fit_config`: Dynamic configuration per round

The server runs for 3 rounds by default.
* The FedAvg strategy in the Server Function.


In [7]:
net = SimpleModel()
params = ndarrays_to_parameters(get_weights(net))

def server_fn(context: Context):
    strategy = FedAvg(
        min_fit_clients=5,
        fraction_evaluate=0.0,
        initial_parameters=params,
        on_fit_config_fn=fit_config,  # <- NEW
    )
    config=ServerConfig(num_rounds=3)
    return ServerAppComponents(
        strategy=strategy,
        config=config,
    )

In [8]:
# Define an instance of ServerApp.
server = ServerApp(server_fn=server_fn)

### Client App

The `FlowerClient` class implements the client-side logic:
- `fit`: Receives global model parameters and config, trains locally, returns updated weights
- `evaluate`: Evaluates the model on local test data


The `client_fn` creates a client instance with its own data partition.

In [9]:
class FlowerClient(NumPyClient):
    def __init__(self, net, trainloader, testloader):
        self.net = net
        self.trainloader = trainloader
        self.testloader = testloader

    def fit(self, parameters, config):
        set_weights(self.net, parameters)

        epochs = config["local_epochs"]
        log(INFO, f"client trains for {epochs} epochs")
        train_model(self.net, self.trainloader, epochs)

        return get_weights(self.net), len(self.trainloader.dataset), {}

    def evaluate(self, parameters, config):
        set_weights(self.net, parameters)
        loss, accuracy = evaluate_model(self.net, self.testloader)
        return loss, len(self.testloader.dataset), {"accuracy": accuracy}

### Create the Client Function and the Client App.

In [10]:
def client_fn(context: Context) -> Client:
    net = SimpleModel()
    partition_id = int(context.node_config["partition-id"])
    trainloader, testloader = load_data(partition_id=partition_id)
    return FlowerClient(net, trainloader, testloader).to_client()


client = ClientApp(client_fn)

### Run Simulation

Finally, we run the federated learning simulation with 5 supernodes (clients).
The `run_simulation` function orchestrates the communication between server and clients.

In [11]:
run_simulation(server_app=server,
               client_app=client,
               num_supernodes=5,
               backend_config=backend_setup)

INFO : Starting Flower ServerApp, config: num_rounds=3, no round_timeout
INFO : 
INFO : [INIT]
INFO : Using initial global parameters provided by strategy
INFO : Starting evaluation of initial global parameters
INFO : Evaluation returned no results (`None`)
INFO : 
INFO : [ROUND 1]
INFO : configure_fit: strategy sampled 5 clients (out of 5)
(ClientAppActor pid=18205) INFO :      client trains for 2 epochs
(ClientAppActor pid=18205) INFO :      Running client 2 epochs
(ClientAppActor pid=18208) INFO :      client trains for 2 epochs [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(ClientAppActor pid=18204) INFO :      Running client 2 epochs [repeated 4x across cluster]
(ClientAppActor pid=18207) INFO :      client trains for 2 epochs
(ClientAppActor pid=18207) INFO :      Running client 2 epochs